In [19]:
# Import libraries

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [20]:
import pandas as pd

df = pd.read_csv(
    "/Users/mukul/ai-ml-internship-portfolio/phase1-data-engineering/data/data/credit_card_fraud_10k.csv"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (10000, 10)


,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,0
1,2,541.82,3,Travel,1,0,87,1,64,0
2,3,237.01,17,Grocery,0,0,49,1,61,0
3,4,164.33,4,Grocery,0,1,72,3,34,0
4,5,30.53,15,Food,0,0,79,0,44,0


In [21]:
# Check dataset

print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

print("\nDuplicate rows:")
print(df.duplicated().sum())

Columns:
['transaction_id', 'amount', 'transaction_hour', 'merchant_category', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age', 'is_fraud']

Missing values:
transaction_id         0
amount                 0
transaction_hour       0
merchant_category      0
foreign_transaction    0
location_mismatch      0
device_trust_score     0
velocity_last_24h      0
cardholder_age         0
is_fraud               0
dtype: int64

Data types:
transaction_id           int64
amount                 float64
transaction_hour         int64
merchant_category          str
foreign_transaction      int64
location_mismatch        int64
device_trust_score       int64
velocity_last_24h        int64
cardholder_age           int64
is_fraud                 int64
dtype: object

Duplicate rows:
0


In [22]:
# Separate features and target

target = "is_fraud"

X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature shape: (10000, 9)
Target shape: (10000,)

Target distribution:
is_fraud
0    9849
1     151
Name: count, dtype: int64


In [23]:
# Remove ID column

X = X.drop(
    columns=["transaction_id"],
    errors="ignore"
)

print("Features after removing transaction ID:")
print(X.columns.tolist())

Features after removing transaction ID:
['amount', 'transaction_hour', 'merchant_category', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']


In [24]:
# Identify feature types

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()



In [25]:
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

/var/folders/1x/3vttmmzn3zg701t67jvcz5hr0000gn/T/ipykernel_9517/304553948.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [26]:
print("Numerical features:")
print(numeric_features)

Numerical features:
['amount', 'transaction_hour', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']


In [27]:
print("\nCategorical features:")
print(categorical_features)


Categorical features:
['merchant_category']


In [28]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data: (8000, 8)
Testing data: (2000, 8)

Training target distribution:
is_fraud
0    7879
1     121
Name: count, dtype: int64

Testing target distribution:
is_fraud
0    1970
1      30
Name: count, dtype: int64


In [29]:
# Numerical preprocessing

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("Numerical preprocessing pipeline created!")

Numerical preprocessing pipeline created!


In [30]:
# Categorical preprocessing

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

print("Categorical preprocessing pipeline created!")

Categorical preprocessing pipeline created!


In [31]:
# Combine numerical and categorical preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("ColumnTransformer created successfully!")

ColumnTransformer created successfully!


In [32]:
# Create complete preprocessing pipeline

preprocessing_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        )
    ]
)

print("Complete preprocessing pipeline created!")

Complete preprocessing pipeline created!


In [33]:
# Fit pipeline using training data

X_train_processed = preprocessing_pipeline.fit_transform(
    X_train
)

print("Training data processed successfully!")
print("Processed training shape:", X_train_processed.shape)

Training data processed successfully!
Processed training shape: (8000, 12)


In [34]:
# Transform test data

X_test_processed = preprocessing_pipeline.transform(
    X_test
)

print("Testing data processed successfully!")
print("Processed testing shape:", X_test_processed.shape)

Testing data processed successfully!
Processed testing shape: (2000, 12)


In [35]:
# Get processed feature names

feature_names = (
    preprocessing_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print("Number of processed features:")
print(len(feature_names))

print("\nProcessed feature names:")
print(feature_names)

Number of processed features:
12

Processed feature names:
['numerical__amount' 'numerical__transaction_hour'
 'numerical__foreign_transaction' 'numerical__location_mismatch'
 'numerical__device_trust_score' 'numerical__velocity_last_24h'
 'numerical__cardholder_age' 'categorical__merchant_category_Clothing'
 'categorical__merchant_category_Electronics'
 'categorical__merchant_category_Food'
 'categorical__merchant_category_Grocery'
 'categorical__merchant_category_Travel']


In [36]:
# Convert processed data into DataFrames

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed training data:")
display(X_train_processed_df.head())

Processed training data:


,numerical__amount,numerical__transaction_hour,numerical__foreign_transaction,numerical__location_mismatch,numerical__device_trust_score,numerical__velocity_last_24h,numerical__cardholder_age,categorical__merchant_category_Clothing,categorical__merchant_category_Electronics,categorical__merchant_category_Food,categorical__merchant_category_Grocery,categorical__merchant_category_Travel
6402,-0.769255,-0.808001,-0.330549,3.280961,0.930866,-0.005472,0.507084,0.0,0.0,1.0,0.0,0.0
5802,-0.677049,-0.808001,-0.330549,-0.304789,-0.930308,-0.005472,-1.696975,0.0,1.0,0.0,0.0,0.0
4131,-0.750482,-1.675479,-0.330549,-0.304789,-1.674777,-0.005472,-1.229448,0.0,0.0,0.0,1.0,0.0
7626,1.267417,1.649853,-0.330549,-0.304789,1.628806,-0.700342,-0.628341,1.0,0.0,0.0,0.0,0.0
6046,-0.888477,-1.675479,-0.330549,-0.304789,1.349630,-0.700342,-1.029079,0.0,0.0,1.0,0.0,0.0


In [37]:
# Check processed data

print("Processed training shape:")
print(X_train_processed_df.shape)

print("\nProcessed testing shape:")
print(X_test_processed_df.shape)

print("\nMissing values after preprocessing:")
print(X_train_processed_df.isna().sum().sum())

print("\nData types:")
print(X_train_processed_df.dtypes.value_counts())

Processed training shape:
(8000, 12)

Processed testing shape:
(2000, 12)

Missing values after preprocessing:
0

Data types:
float64    12
Name: count, dtype: int64


In [39]:
# Test the pipeline on a few random transactions

sample_data = X_test.head(5)

sample_processed = preprocessing_pipeline.transform(
    sample_data
)

print("New data processed successfully!")
print("Input shape:", sample_data.shape)
print("Output shape:", sample_processed.shape)

New data processed successfully!
Input shape: (5, 8)
Output shape: (5, 12)


In [40]:
# Create results folder

os.makedirs("../results", exist_ok=True)

In [41]:
# Save preprocessing pipeline

pipeline_path = "../results/preprocessing_pipeline.pkl"

joblib.dump(
    preprocessing_pipeline,
    pipeline_path
)

print("Pipeline saved successfully!")
print("Location:", pipeline_path)

Pipeline saved successfully!
Location: ../results/preprocessing_pipeline.pkl


In [42]:
# Load saved pipeline

loaded_pipeline = joblib.load(
    "../results/preprocessing_pipeline.pkl"
)

print("Saved pipeline loaded successfully!")

Saved pipeline loaded successfully!


In [43]:
# Test loaded pipeline

test_output = loaded_pipeline.transform(
    X_test.head(5)
)

print("Loaded pipeline works successfully!")
print("Output shape:", test_output.shape)

Loaded pipeline works successfully!
Output shape: (5, 12)


In [44]:
# Save processed datasets

X_train_processed_df.to_csv(
    "../results/X_train_processed.csv",
    index=False
)

X_test_processed_df.to_csv(
    "../results/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../results/y_train.csv",
    index=False
)

y_test.to_csv(
    "../results/y_test.csv",
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [45]:
# Final pipeline check

print("=" * 50)
print("DAY 4 PIPELINE COMPLETED")
print("=" * 50)

print("Raw training data:", X_train.shape)
print("Processed training data:", X_train_processed.shape)

print("Raw testing data:", X_test.shape)
print("Processed testing data:", X_test_processed.shape)

print("\nPipeline:")
print(preprocessing_pipeline)

print("\nPipeline saved at:")
print("../results/preprocessing_pipeline.pkl")

DAY 4 PIPELINE COMPLETED
Raw training data: (8000, 8)
Processed training data: (8000, 12)
Raw testing data: (2000, 8)
Processed testing data: (2000, 12)

Pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['amount', 'transaction_hour',
                                                   'foreign_transaction',
                                                   'location_mismatch',
                                                   'device_trust_score',
                                                   'velocity_last_24h',
                       